# Visual Attribute Extraction

**Analogue of:** `attribute_extraction.ipynb` (§4.2 of Patchscopes paper)

**Research question:** Can we decode specific visual attributes (color, shape, object category,
spatial relationship, material) from a visual patch token's hidden state — without training
any probes?

**Method:** Zero-Shot Visual Feature Extraction Patchscope  
- Source: image + text prompt; extract visual patch token hidden state  
- Target: attribute-specific verbalization prompt (e.g. `'The color of the object in this region is:'`)  
- Score: does `ground_truth` appear in the generated text?  

**Expected finding:** Patchscopes outperforms linear probes in early/mid LLM backbone layers
(same pattern as text). Strongly outperforms Logit Lens because visual tokens were not
trained to be decoded via the unembedding matrix.

**Dataset needed:** GQA or Visual Genome (image, region, attribute_type, value) triplets  
- GQA: https://cs.stanford.edu/people/dorarad/gqa/about.html  
- Visual Genome: https://visualgenome.org/


In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from general_utils import ModelAndTokenizer
from patchscopes_utils import (
    set_hs_patch_hooks_llava_batch,
    evaluate_visual_attribute_extraction_batch,
)

## 1. Load Model

In [ ]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

import torch
mt = ModelAndTokenizer(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device="cuda",
)
mt.set_hs_patch_hooks = set_hs_patch_hooks_llava_batch

print(mt)
print(f"is_vlm={mt.is_vlm}, num_layers={mt.num_layers}")

## 2. Target Prompts per Attribute Type

Each attribute type gets its own verbalization prompt, analogous to the relation-specific
prompts used in the text attribute extraction experiment.

In [ ]:
ATTRIBUTE_TARGET_PROMPTS = {
    "color":    "The color of the object in this region is:",
    "category": "The object shown here is a type of:",
    "material": "The material this object is made of is:",
    "shape":    "The shape of this object is:",
    "location": "This object is located in the:",
    "relation": "The spatial relationship shown here is:",
}

SOURCE_PROMPT = "USER: Describe this image. ASSISTANT:"

## 3. Load Dataset

Each row: `image_path`, `patch_index`, `attribute_type`, `ground_truth`

In [ ]:
# TODO: set path to your GQA / Visual Genome attribute dataset
DATASET_PATH = "./preprocessed_data/visual_attributes.tsv"

dataset_df = pd.read_csv(DATASET_PATH, sep="\t")
print(f"Loaded {len(dataset_df)} samples")
print("Attribute types:", dataset_df["attribute_type"].unique())
dataset_df.head()

## 4. Build Experiment DataFrame

One row per (sample, layer_source, layer_target) combination.

In [ ]:
rows = []
for _, row in dataset_df.iterrows():
    target_prompt = ATTRIBUTE_TARGET_PROMPTS.get(row["attribute_type"],
                        "The attribute of the object shown here is:")
    for l_s in range(mt.num_layers):
        for l_t in range(mt.num_layers):
            rows.append({
                "image_path": row["image_path"],
                "prompt_source": SOURCE_PROMPT,
                "prompt_target": target_prompt,
                "position_source": int(row["patch_index"]),
                "position_target": -1,
                "layer_source": l_s,
                "layer_target": l_t,
                "modality": "visual",
                "ground_truth": row["ground_truth"],
                "attribute_type": row["attribute_type"],
            })

exp_df = pd.DataFrame(rows)
print(f"Experiment rows: {len(exp_df)}")

## 5. Run Evaluation

In [ ]:
results = evaluate_visual_attribute_extraction_batch(
    mt,
    exp_df,
    batch_size=32,
    max_gen_len=10,
    transform=None,
)

exp_df["generation"] = results["generations"]
exp_df["is_correct"] = results["is_correct"]

exp_df.to_csv("./results_visual_attribute_extraction.csv", index=False)
print("Saved results.")

## 6. Visualize

Heatmap of (layer_source × layer_target) accuracy per attribute type.

In [ ]:
attribute_types = exp_df["attribute_type"].unique()
n_cols = 3
n_rows = int(np.ceil(len(attribute_types) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = axes.flatten()

for idx, attr in enumerate(attribute_types):
    subset = exp_df[exp_df["attribute_type"] == attr]
    pivot = subset.groupby(["layer_source", "layer_target"])["is_correct"].mean().unstack()
    sns.heatmap(
        pivot, ax=axes[idx], cmap="RdYlGn", vmin=0, vmax=1,
        cbar_kws={"label": "Accuracy"},
    )
    axes[idx].set_title(f"{attr}")
    axes[idx].set_xlabel("layer_target")
    axes[idx].set_ylabel("layer_source")

for idx in range(len(attribute_types), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Visual Attribute Extraction Accuracy (layer_source × layer_target)", y=1.01)
plt.tight_layout()
plt.savefig("./visual_attribute_extraction_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Summary: mean accuracy per attribute type
summary = exp_df.groupby("attribute_type")["is_correct"].mean().sort_values(ascending=False)
print("Mean accuracy per attribute type:")
print(summary.to_string())